# Core Concepts 

Dive deep into LangGraph's core components.

## Nodes in LangGraph

Nodes are the building blocks of a graph.

You can think of a node as:
- a step
- a task
- or a function

Each node performs some work inside the workflow.

Example:
- taking user input
- calling an AI model
- using a tool
- saving memory
- making decisions

In simple words:

One node = One job

### Implementation

In [2]:
def process_input(state):
    state["processed"] = state["input"].upper()
    return state

state = {
    "input":"hello",
    "processed": ""
    }

new_state = process_input(state)
print(new_state)    

{'input': 'hello', 'processed': 'HELLO'}


input->process->updated state->output

## Edges
Edges define the flow between nodes.

### Types:
- **Normal Edges**: Always follow
- **Conditional Edges**: Based on conditions

They can be unconditional or conditional.

## State
State is shared memory across the graph.

### Best Practices:
- Use dictionaries
- Keep it serializable
- Update immutably

It's a dictionary that persists between nodes.

In [4]:
initial_state = {
    "messages": [], # list to store the conversation history
    "memory":{}, # dict to store any relevant information we want to keep track of
    "counter": 0 # keep track of how many steps have happened
}

def update_state(state):
    # Update conversation history
    state["counter"] += 1 # increment step counter
    state["messages"].append(f"Step{state['counter']}") # add a new msg showing current step number
    return state
print(update_state(initial_state.copy()))

{'messages': ['Step1'], 'memory': {}, 'counter': 1}


## Conditional Routing
Routes based on conditions, like if-else flows.

### Examples:
- If query contains 'help' → FAQ node
- If confidence < 0.5 → Human review

In [6]:
def decide_route(state):
    if "tool" in state["input"].lower():
        return "tool_node"
    elif "help" in state["input"].lower():
        return "faq_node"
    else:
        return "llm_node"

# Test
print(decide_route({"input": "I need a tool"}))
print(decide_route({"input": "Help me"}))
print(decide_route({"input": "Hello"}))

tool_node
faq_node
llm_node


## Loops
Allow retries or iterative processes.

### Use Cases:
- Retry on failure
- Refine answers
- Multi-step reasoning

## Demo: Simple Q&A Agent with Memory
Build a basic agent that answers questions and remembers context.

In [7]:
from dotenv import load_dotenv
import os

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

print(bool(groq_api_key))

True


In [8]:

from langgraph.graph import StateGraph
from langchain_groq import ChatGroq


# Initialize Groq LLM
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.1-8b-instant",
    temperature=0
)

# Define node
def llm_node(state):
    user_message = state["messages"][-1]

    # Get response from Groq
    response = llm.invoke(user_message)

    # Save response
    state["messages"].append(response.content)
    state["memory"]["last_query"] = user_message

    return state

# Create graph
graph = StateGraph(dict)

graph.add_node("llm", llm_node) # add llm node to graph
graph.set_entry_point("llm")  # set starting point of graph
graph.set_finish_point("llm") # graph stops after this node


# Compile graph
app = graph.compile()

# Initial state
initial_state = {
    "messages": ["What is AI?"],
    "memory": {}
}

# Run graph
result = app.invoke(initial_state)

print(result)


{'messages': ['What is AI?', 'AI, or Artificial Intelligence, refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. The term can also be applied to any machine that exhibits traits associated with a human mind such as learning and problem-solving.\n\nAI technology is based on the principle of creating algorithms that can process data, identify patterns, and make decisions with minimal human intervention. This is achieved through various techniques such as machine learning, natural language processing, and deep learning.\n\nThere are several types of AI, including:\n\n1. **Narrow or Weak AI**: This type of AI is designed to perform a specific task, such as facial recognition, language translation, or playing chess. Narrow AI is trained on a specific dataset and is not capable of general reasoning or problem-solving.\n2. **General or Strong AI**: This type of AI is designed to perform any intellectual task that a human can. General

## Exercises: Build Your Own Simple Graph
Create a graph with 3 nodes: input processor, decision maker, and responder.

In [9]:
#input -> processing ->decision -> diffrenciate outpu -> end

# Exercise: Build a simple graph
# 1. Define state
# 2. Create nodes
# 3. Add edges
# 4. Compile and run


from langgraph.graph import StateGraph, END

# Nodes
def input_processor(state):
    state["processed"] = state["input"].strip().lower()
    return state

def decision_maker(state):
    # Just return state
    return state

def route_decision(state):
    if len(state["processed"]) > 10:
        return "long"
    else:
        return "short"

def long_responder(state):
    state["response"] = "Long answer for: " + state["processed"]
    return state

def short_responder(state):
    state["response"] = "Short: " + state["processed"]
    return state

# Graph
graph = StateGraph(dict)

graph.add_node("input", input_processor)
graph.add_node("decide", decision_maker)
graph.add_node("long", long_responder)
graph.add_node("short", short_responder)

# Flow
graph.set_entry_point("input")

graph.add_edge("input", "decide")

# Conditional routing
graph.add_conditional_edges(
    "decide",
    route_decision,
    {
        "long": "long",
        "short": "short"
    }
)

# End nodes
graph.add_edge("long", END)
graph.add_edge("short", END)

# Compile
app = graph.compile()

# Run
result = app.invoke({
    "input": "This is a long question"
})

print(result)


{'input': 'This is a long question', 'processed': 'this is a long question', 'response': 'Long answer for: this is a long question'}


input ->process->decide ->branch ->output

## Summary
- Nodes: Functions that process state
- Edges: Control flow
- State: Shared memory
- Conditionals: Branching logic
- Loops: Iterative processes

